In [ ]:
#q2 bernoulli naive bayes bayesiano
# Levi Félix Barbosa - 579420

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv("cleaned_reviews.csv").dropna()

le = LabelEncoder()
y_all = le.fit_transform(data.sentiments)
txt = data.cleaned_review.values
C = len(le.classes_)

print(data.shape)
print(dict(zip(le.classes_, np.bincount(y_all))))

(17337, 4)
{'negative': np.int64(1534), 'neutral': np.int64(6300), 'positive': np.int64(9503)}


In [ ]:
def fit(X, y, a, b, alpha=1.0):
    Nc = np.bincount(y, minlength=C)
    # Ndc[c, d] = em quantos docs da classe c a palavra d aparece
    Ndc = np.vstack([np.asarray(X[y == c].sum(axis=0)).ravel() for c in range(C)])

    # medias das posterioris beta e dirichlet
    theta = (Ndc + a) / (Nc[:, None] + a + b)
    pi = (Nc + alpha) / (len(y) + C * alpha)
    return pi, theta


def predict(X, pi, theta):
    lt = np.log(theta)
    l1t = np.log1p(-theta)
    base = np.log(pi) + l1t.sum(axis=1)
    scores = np.asarray(X @ (lt - l1t).T) + base
    return scores.argmax(axis=1)

In [4]:
prioris = [(1.0, 1.0), (0.5, 0.5), (2.0, 2.0)]
acc = {p: [] for p in prioris}

rng = np.random.default_rng(42)

for r in range(20):
    idx = rng.permutation(len(y_all))
    tr, te = idx[:1000], idx[1000:]

    # vocabulario so com o treino, senao vaza info do teste
    enc = CountVectorizer(binary=True).fit(txt[tr])
    Xtr, Xte = enc.transform(txt[tr]), enc.transform(txt[te])

    # as tres prioris usam o mesmo split da rodada
    for a, b in prioris:
        pi, theta = fit(Xtr, y_all[tr], a, b)
        acc[(a, b)].append((predict(Xte, pi, theta) == y_all[te]).mean())

print("vocabulario da ultima rodada:", Xtr.shape[1])
print("teste:", Xte.shape[0])

vocabulario da ultima rodada: 2740
teste: 16337


In [5]:
A = {p: np.array(v) for p, v in acc.items()}

print("priori            media      dp")
for (a, b), v in A.items():
    print(f"Beta({a}, {b})    {v.mean():.4f}     {v.std():.4f}")

print(f"\nchutar sempre a classe majoritaria daria {np.bincount(y_all).max() / len(y_all):.4f}")

priori            media      dp
Beta(1.0, 1.0)    0.6592     0.0288
Beta(0.5, 0.5)    0.6411     0.0239
Beta(2.0, 2.0)    0.6918     0.0268

chutar sempre a classe majoritaria daria 0.5481


In [ ]:
# comparo rodada a rodada pra ver se a diferenca é real
d1 = A[(2.0, 2.0)] - A[(1.0, 1.0)]
d2 = A[(1.0, 1.0)] - A[(0.5, 0.5)]

print(f"Beta(2,2) > Beta(1,1)     em {(d1 > 0).sum()}/20 rodadas")
print(f"Beta(1,1) > Beta(0.5,0.5) em {(d2 > 0).sum()}/20 rodadas")

# ganha em todas
# faz sentido: com 1000 docs o vocabulario passa de 2700 palavras e muita delas
# quase nao aparecem, ai quem decide o theta é a priori.

Beta(2,2) > Beta(1,1)     em 20/20 rodadas
Beta(1,1) > Beta(0.5,0.5) em 20/20 rodadas
